<a href="https://colab.research.google.com/github/amann45/AI_Labworks/blob/main/Lab7/lab7RNN_Encoder_Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN Encoder-Decoder [Aman Kumar Ray: ACE080BCT010]



## Objective
To implement and train a basic Recurrent Neural Network (RNN) Encoder-Decoder model for sequence-to-sequence translation, specifically for translating short French phrases into English.

## Theory
The Encoder–Decoder architecture, also known as the **Sequence-to-Sequence (Seq2Seq)** model, is a deep learning framework designed to transform one sequence into another sequence. It is widely used in Natural Language Processing (NLP) tasks such as machine translation, text summarization, speech recognition, question answering, and conversational AI.

The architecture consists of two Recurrent Neural Networks (RNNs):

1. **Encoder**
2. **Decoder**

The encoder reads the input sequence one token at a time and compresses the information into a fixed-length context vector. The decoder then uses this context vector to generate the corresponding output sequence one token at a time.

---

## Working Principle

The Seq2Seq model follows these steps:

1. The input sentence is tokenized into individual words.
2. Each word is converted into its numerical representation (index or embedding).
3. The encoder processes each input word sequentially and updates its hidden state.
4. After reading the complete input sentence, the encoder produces a context vector representing the entire sentence.
5. This context vector is passed to the decoder.
6. The decoder starts with a special **Start-of-Sequence (SOS)** token and predicts one word at a time.
7. The prediction continues until an **End-of-Sequence (EOS)** token is generated.

-----

## Sequence-to-Sequence Architecture Diagram

```
             Input Sentence
          "I am a student"
                  │
                  ▼
        +-------------------+
        |      Encoder      |
        | (RNN/LSTM/GRU)    |
        +-------------------+
                  │
                  ▼
           Context Vector
                  │
                  ▼
        +-------------------+
        |      Decoder      |
        | (RNN/LSTM/GRU)    |
        +-------------------+
                  │
                  ▼
      "Je suis un étudiant"

In [26]:
# requirements
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [27]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [28]:
# Turn a Unicode string to plain ASCII, thanks to
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [29]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [30]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [31]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [32]:
PATH = r'/content/data.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['je suis degoute', 'i m disgusted']


15

### Encoder
The encoder is responsible for understanding the input sentence. It consists of an RNN, LSTM, or GRU network that processes each input token sequentially.

For each input word:

- The current word embedding is taken as input.
- The hidden state from the previous time step is updated.
- Information from previous words is retained in the hidden state.

After processing the final word, the last hidden state becomes the **context vector**, which summarizes the entire input sentence.

Mathematically,

\[
h_t = f(x_t, h_{t-1})
\]

where:

- \(x_t\) = input at time step \(t\)
- \(h_{t-1}\) = previous hidden state
- \(h_t\) = current hidden state
- \(f\) = RNN/LSTM/GRU function

In [33]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

### Decoder
The decoder generates the target sequence using the context vector received from the encoder.

At each decoding step:

- The previous output word is used as the next input.
- The hidden state is updated.
- A probability distribution over the vocabulary is produced.
- The word with the highest probability is selected.

The process continues until the decoder predicts the EOS (End-of-Sequence) token.

In [34]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

## Training
### Data Preparation

Before training the Seq2Seq model, the dataset must be preprocessed.

The preprocessing steps include:

- Reading the parallel language dataset.
- Splitting sentence pairs.
- Converting all text to lowercase.
- Removing unwanted characters and punctuation.
- Tokenizing each sentence.
- Creating a vocabulary.
- Assigning an integer index to every unique word.
- Filtering very long sentences to reduce training complexity.

The vocabulary usually contains:

- **word2index** : maps each word to a unique integer.
- **index2word** : converts integers back into words.
- **word2count** : stores the frequency of each word.

### During training

1. The encoder processes the input sentence.
2. The final hidden state is passed to the decoder.
3. The decoder predicts the next target word.
4. The predicted output is compared with the actual target sentence.
5. The loss is computed using Cross-Entropy Loss.
6. Backpropagation Through Time (BPTT) updates the network parameters using an optimizer such as Adam.

This process is repeated for multiple epochs until the model converges.



In [35]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [36]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [37]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [38]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [39]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [40]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [41]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [42]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 12s (- 8m 15s) (5 2%) 1.9980
0m 24s (- 7m 52s) (10 5%) 1.3081
0m 37s (- 7m 41s) (15 7%) 1.1157
0m 51s (- 7m 39s) (20 10%) 0.9813
1m 3s (- 7m 23s) (25 12%) 0.8730
1m 15s (- 7m 9s) (30 15%) 0.7766
1m 28s (- 6m 55s) (35 17%) 0.6873
1m 40s (- 6m 42s) (40 20%) 0.6019
1m 53s (- 6m 30s) (45 22%) 0.5286
2m 6s (- 6m 18s) (50 25%) 0.4645
2m 18s (- 6m 5s) (55 27%) 0.4111
2m 30s (- 5m 51s) (60 30%) 0.3619
2m 43s (- 5m 38s) (65 32%) 0.3227
2m 55s (- 5m 25s) (70 35%) 0.2854
3m 6s (- 5m 11s) (75 37%) 0.2510
3m 19s (- 4m 58s) (80 40%) 0.2233
3m 32s (- 4m 47s) (85 42%) 0.2038
3m 45s (- 4m 35s) (90 45%) 0.1794
3m 58s (- 4m 24s) (95 47%) 0.1631
4m 12s (- 4m 12s) (100 50%) 0.1518
4m 25s (- 4m 0s) (105 52%) 0.1371
4m 41s (- 3m 50s) (110 55%) 0.1245
4m 54s (- 3m 37s) (115 57%) 0.1147
5m 7s (- 3m 24s) (120 60%) 0.1106
5m 20s (- 3m 12s) (125 62%) 0.0997
5m 32s (- 2m 59s) (130 65%) 0.

In [43]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> il est trop saoul
= he s too drunk
< he s too drunk <EOS>

> nous sommes attaquees
= we re being attacked
< i m a salesperson <EOS>

> nous sommes des hommes
= we are men
< we are men <EOS>

> vous etes tres astucieux
= you re very astute
< you re very astute <EOS>

> vous etes sauf
= you re safe
< he s a senior <EOS>

> je suis vanne
= i m exhausted
< you re all right <EOS>

> tu es trop lent
= you re too slow
< you re too slow <EOS>

> je suis vraiment desole
= i am truly sorry
< i m very sorry <EOS>

> t es plantee
= you re stuck
< she is my girlfriend <EOS>

> je suis etudiant
= i m a student
< you re all happy <EOS>



## Discussion

In this lab, the Encoder–Decoder (Seq2Seq) model was successfully implemented to perform sequence-to-sequence learning. The encoder converted the input sentence into a context vector, while the decoder generated the corresponding output sequence. The experiment demonstrated the complete workflow of data preprocessing, training, and prediction. It also showed that translation quality depends on the amount of training data and model parameters, with performance decreasing for longer sequences due to the fixed-length context vector.

## Conclusion

This lab successfully demonstrated the working of the Encoder–Decoder architecture for machine translation. The model effectively learned to map input sequences to output sequences and provided a practical understanding of sequence-to-sequence learning. Overall, the lab highlighted the importance of Encoder–Decoder networks in Natural Language Processing and their applications in translation, chatbots, and text generation.